# MASTER · SEPS Segmento 1 Risk
Orquestador único para Colab. **Update 015 · v0.15.0**

**Arquitectura:** Parquet mensual particionado = almacenamiento canónico; DuckDB = motor analítico efímero.

**Dashboard 015:** conserva la UX 014 y añade flechas semánticas en los KPI superiores respecto al período anterior según la agregación seleccionada. Incluye además una celda final opcional para publicar el proyecto completo en GitHub sin bases ni secretos.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
from pathlib import Path
import os, sys
exact = list(Path('/content/drive/MyDrive').rglob('seps-segmento1-risk/config/project.yaml'))
fallback = list(Path('/content/drive/MyDrive').rglob('config/project.yaml')) if not exact else []
candidates = exact or fallback
assert candidates, 'No encontré config/project.yaml en Google Drive'
PROJECT_ROOT = candidates[0].parents[1]
os.chdir(PROJECT_ROOT)
SRC = PROJECT_ROOT / 'src'
assert (SRC / 'sepsrisk' / '__init__.py').exists(), f'Proyecto incompleto: no existe {SRC / "sepsrisk"}'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('PROJECT_ROOT:', PROJECT_ROOT)
print('SRC:', SRC)


PROJECT_ROOT: /content/drive/MyDrive/seps-segmento1-risk
SRC: /content/drive/MyDrive/seps-segmento1-risk/src


In [3]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)])
import sepsrisk
print('sepsrisk:', sepsrisk.__file__)
print('versión:', getattr(sepsrisk, '__version__', 'n/d'))


sepsrisk: /content/drive/MyDrive/seps-segmento1-risk/src/sepsrisk/__init__.py
versión: 0.15.0


## Pipeline completo
Discovery cacheado/incremental, resume por SHA, Parquet mensual, DuckDB, QA, features de todo Segmento 1, señales, inferencia, dashboard multi-entidad y Atlas/Handoff.

**015:** las tarjetas KPI muestran flecha ↑/↓ respecto al período anterior; el color indica si el movimiento es favorable (verde) o adverso (rojo). La comparación respeta la agregación seleccionada: mes, trimestre, semestre o año.

El dashboard sigue siendo completamente estático: no hace polling ni autoreload.


In [4]:
from sepsrisk.pipeline import run
summary = run(PROJECT_ROOT, ingest=True, show_progress=True)
summary

SEPS · preparando:   0%|          |   0.0/100% [00:00<?, ?%/s] 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/content/drive/MyDrive/seps-segmento1-risk/src/sepsrisk/features/engine.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m &= eeff.account_description.fillna('').str.contains(rx,regex=True,na=False)
/content/drive/MyDrive/seps-segmento1-risk/src/sepsrisk/features/engine.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m &= eeff.account_description.fillna('').str.contains(rx,regex=True,na=False)
/content/drive/MyDrive/seps-segmento1-risk/src/sepsrisk/features/engine.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m &= eeff.account_description.fillna('').str.contains(rx,regex=True,na=False)
/content/drive/MyDrive/seps-segmento1-risk/src/sepsrisk/features/engine.py:19: UserWarning: This pattern is interpreted as a

{'started_at': '2026-09-21T17:44:20.253334+00:00',
 'backend': 'parquet+duckdb',
 'sources_checked': 10,
 'sources_discovered': 10,
 'inserted': 0,
 'updated': 0,
 'unchanged': 0,
 'resume_skipped': 8,
 'source_sha_skipped': 0,
 'errors': [{'source': 'EEFF-MEN-BASE-2026',
   'error': 'ConnectTimeout(MaxRetryError("HTTPSConnectionPool(host=\'estadisticas.seps.gob.ec\', port=443): Max retries exceeded with url: /?sdm_process_download=1&download_id=3258 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x794d32c39bd0>, \'Connection to estadisticas.seps.gob.ec timed out. (connect timeout=90)\'))"))'},
  {'source': 'EEFF-MEN-BASE-ANTERIORES',
   'error': 'ConnectTimeout(MaxRetryError("HTTPSConnectionPool(host=\'estadisticas.seps.gob.ec\', port=443): Max retries exceeded with url: /?sdm_process_download=1&download_id=899 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x794d32c39f90>, \'Connection to estadisticas.seps.gob.ec timed out. 

In [5]:
print('Dashboard:', PROJECT_ROOT / 'githubpage/index.html')
print('Estado:', PROJECT_ROOT / 'atlas/latest_state.json')
print('Atlas:', PROJECT_ROOT / 'atlas/atlas.html')
print('Handoff:', PROJECT_ROOT / 'atlas/handoff/LATEST_STATE.zip')
print('Atlas unresolved:', summary.get('atlas', {}).get('unresolved'))


Dashboard: /content/drive/MyDrive/seps-segmento1-risk/githubpage/index.html
Estado: /content/drive/MyDrive/seps-segmento1-risk/atlas/latest_state.json
Atlas: /content/drive/MyDrive/seps-segmento1-risk/atlas/atlas.html
Handoff: /content/drive/MyDrive/seps-segmento1-risk/atlas/handoff/LATEST_STATE.zip
Atlas unresolved: 0


## Publicar el proyecto de código en GitHub
Repositorio configurado: `mathematicatutorias-ai/seps-segmento1-risk-code` · rama `main`.

La celda publica código, configuración, contratos, tests, notebooks, Atlas liviano y `githubpage/`. **Nunca publica Parquet, SQLite, DuckDB, bases operativas ni secretos.**

Antes de usarla una vez, crea en Colab un Secret llamado **`GITHUB_TOKEN`** con un Fine-grained Personal Access Token limitado a este repositorio y permiso **Contents: Read and write**. Activa `PUBLISH_GITHUB=True` y ejecuta solo esta celda cuando quieras publicar.


In [20]:
# PUBLISH_GITHUB = False  # Cambia a True y ejecuta esta celda para publicar.
PUBLISH_GITHUB = True

if PUBLISH_GITHUB:
    from sepsrisk.publication.github import publish_project
    github_result = publish_project(PROJECT_ROOT, commit_message='Release 015')
    print('GitHub:', github_result['status'])
    print('Repo:', github_result['remote'])
    print('Rama:', github_result['branch'])
    print('Commit:', github_result.get('commit', '—'))
    print('Archivos publicados:', github_result['files'])
else:
    print('Publicación GitHub desactivada. Define PUBLISH_GITHUB=True para subir el proyecto.')


RuntimeError: git push falló: remote: Permission to mathematicatutorias-ai/seps-segmento1-risk-code.git denied to mathematicatutorias-ai.
fatal: unable to access 'https://github.com/mathematicatutorias-ai/seps-segmento1-risk-code.git/': The requested URL returned error: 403